In [ ]:
# Importe
from pathlib import Path
import json
import platform
import warnings
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
try:
    import pm4py
except ImportError as e:
    raise ImportError('PM4Py fehlt. Bitte in der .venv installieren: pip install -U pm4py') from e
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, balanced_accuracy_score, accuracy_score, brier_score_loss, log_loss, confusion_matrix, roc_curve, precision_recall_curve
print('Notebook 06 läuft.')
print('Python:', platform.python_version())
print('Platform:', platform.platform())


In [ ]:
# Pfade und Einstellungen
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'prediction_refinement_ablation_timeprefixes'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_JOBS = -1
PRIMARY_LABEL = 'label_scd_p90_or_global'
ROBUSTNESS_LABELS = ['label_scd_p90_or_yearnorm', 'label_scd_p90_or_no_inspection_global', 'label_scd_p90_or_no_inspection_yearnorm', 'label_path_change_or_objection', 'label_temporal_duration_p90_yearnorm']
TOP_N_RAW_ACTIVITIES = 30
TOP_N_COMBINED_ACTIVITIES = 60
TOP_N_RESOURCES = 30
MIN_CASES_PER_CATEGORY = 50
MIN_CASES_PER_DEPT_THRESHOLD = 100
EVENT_PREFIX_SPECS = [{'prefix_id': 'event20_all', 'prefix_type': 'event', 'value': 20, 'require_min_events': False}, {'prefix_id': 'event20_min20', 'prefix_type': 'event', 'value': 20, 'require_min_events': True}]
TIME_PREFIX_SPECS = [{'prefix_id': 'time7d', 'prefix_type': 'time_days', 'value': 7, 'require_min_events': False}, {'prefix_id': 'time30d', 'prefix_type': 'time_days', 'value': 30, 'require_min_events': False}, {'prefix_id': 'time60d', 'prefix_type': 'time_days', 'value': 60, 'require_min_events': False}, {'prefix_id': 'time90d', 'prefix_type': 'time_days', 'value': 90, 'require_min_events': False}]
PRIMARY_PREFIX_SPECS = EVENT_PREFIX_SPECS + TIME_PREFIX_SPECS
ROBUSTNESS_PREFIX_SPECS = [{'prefix_id': 'event20_all', 'prefix_type': 'event', 'value': 20, 'require_min_events': False}, {'prefix_id': 'time30d', 'prefix_type': 'time_days', 'value': 30, 'require_min_events': False}, {'prefix_id': 'time60d', 'prefix_type': 'time_days', 'value': 60, 'require_min_events': False}]
RUN_DUMMY_BASELINE = True
RUN_LOGISTIC_REGRESSION = True
RUN_RANDOM_FOREST = True
RF_N_ESTIMATORS = 180
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 5
MODELS_MAIN = []
if RUN_DUMMY_BASELINE:
    MODELS_MAIN.append('dummy_prior')
if RUN_LOGISTIC_REGRESSION:
    MODELS_MAIN.append('logreg_balanced')
if RUN_RANDOM_FOREST:
    MODELS_MAIN.append('rf_balanced')
ROBUSTNESS_MODELS = [m for m in ['logreg_balanced', 'rf_balanced'] if m != 'rf_balanced' or RUN_RANDOM_FOREST]
ABLATION_SCENARIOS = {'full_allowed': {'description': 'Alle leakage-erlaubten frühen Features.', 'drop_patterns': [], 'keep_only_prefix_features': False}, 'no_department': {'description': 'Entfernt Department-/Abteilungsinformationen.', 'drop_patterns': ['department', 'dept'], 'keep_only_prefix_features': False}, 'no_case_year': {'description': 'Entfernt case:year und start_year-Kontext, um Jahres-/Drift-Signal zu testen.', 'drop_patterns': ['case:year', 'case_year', 'start_year', 'year_from_timestamp'], 'keep_only_prefix_features': False}, 'no_resource': {'description': 'Entfernt Resource-ID- und Resource-Count-Features.', 'drop_patterns': ['resource', 'org:resource', 'org_resource'], 'keep_only_prefix_features': False}, 'no_inspection_features': {'description': 'Entfernt Features, die Inspection-/On-Site-Kontext direkt anzeigen.', 'drop_patterns': ['inspection', 'on-site', 'on_site', 'onsite'], 'keep_only_prefix_features': False}, 'prefix_only_no_static': {'description': 'Nur Prefix-Features; statische Start-/Case-Attribute werden entfernt.', 'drop_patterns': [], 'keep_only_prefix_features': True}, 'no_dept_resource_inspection': {'description': 'Strenge Ablation: ohne Department, Resource und Inspection-Signale.', 'drop_patterns': ['department', 'dept', 'resource', 'org:resource', 'org_resource', 'inspection', 'on-site', 'on_site', 'onsite'], 'keep_only_prefix_features': False}}
BASELINE_05_TABLE_DIR = PROJECT_ROOT / 'outputs' / 'prediction_design_baseline_modeling' / 'tables'
if not LOG_PATH.exists():
    candidates = sorted(DATA_RAW.glob('*.xes*')) + sorted(DATA_RAW.glob('**/*.xes*'))
    print('Gefundene XES-Kandidaten:')
    for c in candidates[:20]:
        print('-', c)
    if candidates:
        LOG_PATH = candidates[0]
        print('Nutze automatisch:', LOG_PATH)
    else:
        raise FileNotFoundError(f'Keine XES/XES.GZ-Datei in {DATA_RAW} gefunden.')
print('Project root:', PROJECT_ROOT)
print('Log path:', LOG_PATH)
print('Output root:', OUTPUT_ROOT)
print('Primary prefix specs:', [p['prefix_id'] for p in PRIMARY_PREFIX_SPECS])
print('Ablation scenarios:', list(ABLATION_SCENARIOS.keys()))
print('Models:', MODELS_MAIN)


In [ ]:
# Hilfsfunktionen
created_tables = []
created_figures = []
analysis_notes = []

def save_csv(obj, filename, index=True):
    path = TABLE_DIR / filename
    if isinstance(obj, pd.Series):
        obj.to_frame().to_csv(path, index=index, encoding='utf-8-sig')
    else:
        obj.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def robust_to_bool(s, default=False):
    if isinstance(s, pd.Series):
        if s.dtype == bool:
            return s.fillna(default).astype(bool)
        s_str = s.astype(str).str.strip().str.lower()
        true_values = {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}
        false_values = {'false', '0', '0.0', 'no', 'n', 'nein', 'falsch', 'nan', 'none', '<na>', '', 'f'}
        out = pd.Series(bool(default), index=s.index)
        out[s_str.isin(true_values)] = True
        out[s_str.isin(false_values)] = False
        numeric = pd.to_numeric(s, errors='coerce')
        out[numeric.fillna(0) > 0] = True
        return out.astype(bool)
    if pd.isna(s):
        return bool(default)
    if isinstance(s, str):
        return s.strip().lower() in {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}
    return bool(s)

def numeric_series(df_, col, default=0):
    if col in df_.columns:
        return pd.to_numeric(df_[col], errors='coerce').fillna(default)
    return pd.Series(default, index=df_.index)

def quantile(series, q):
    return float(pd.to_numeric(series, errors='coerce').quantile(q))

def sanitize_col_name(value):
    s = str(value)
    s = s.strip()
    s = re.sub('[^0-9a-zA-ZäöüÄÖÜß]+', '_', s)
    s = re.sub('_+', '_', s)
    s = s.strip('_')
    if not s:
        s = 'missing'
    return s[:90]

def top_values_from_train(prefix_events, col, train_cases, top_n):
    if col not in prefix_events.columns:
        return []
    sub = prefix_events[prefix_events[CASE_COL].isin(train_cases)]
    vc = sub[col].astype(str).value_counts(dropna=False)
    return vc.head(top_n).index.tolist()

def safe_auc(metric_func, y_true, y_score):
    try:
        y_true = np.asarray(y_true).astype(int)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(metric_func(y_true, y_score))
    except Exception:
        return np.nan

def safe_log_loss(y_true, y_score):
    try:
        y_true = np.asarray(y_true).astype(int)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(log_loss(y_true, y_score, labels=[0, 1]))
    except Exception:
        return np.nan

def select_threshold_by_f1(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return (0.5, np.nan)
    thresholds = np.unique(np.quantile(y_score, np.linspace(0.01, 0.99, 99)))
    best_thr = 0.5
    best_f1 = -1
    for thr in thresholds:
        pred = y_score >= thr
        f = f1_score(y_true, pred, zero_division=0)
        if f > best_f1:
            best_f1 = f
            best_thr = float(thr)
    return (best_thr, float(best_f1))

def evaluate_scores(y_true, y_score, threshold, target, prefix_id, scenario, model_name, split_name, threshold_policy):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = y_score >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {'target': target, 'prefix_id': prefix_id, 'scenario': scenario, 'model': model_name, 'split': split_name, 'threshold_policy': threshold_policy, 'threshold': float(threshold), 'n': int(len(y_true)), 'positive_cases': int(y_true.sum()), 'prevalence_pct': float(y_true.mean() * 100) if len(y_true) else np.nan, 'roc_auc': safe_auc(roc_auc_score, y_true, y_score), 'pr_auc_average_precision': safe_auc(average_precision_score, y_true, y_score), 'precision': float(precision_score(y_true, y_pred, zero_division=0)), 'recall': float(recall_score(y_true, y_pred, zero_division=0)), 'f1': float(f1_score(y_true, y_pred, zero_division=0)), 'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)), 'accuracy': float(accuracy_score(y_true, y_pred)), 'brier_score': float(brier_score_loss(y_true, y_score)) if len(y_true) else np.nan, 'log_loss': safe_log_loss(y_true, y_score), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=MIN_CASES_PER_CATEGORY)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def get_feature_names_from_pipeline(pipe):
    try:
        pre = pipe.named_steps['preprocess']
        return list(pre.get_feature_names_out())
    except Exception:
        return []
print('Helper geladen.')


In [ ]:
# Log laden
print('Lade XES-Log ...')
log = pm4py.read_xes(str(LOG_PATH))
event_df = pm4py.convert_to_dataframe(log)
print('Event DataFrame:', event_df.shape)
CASE_COL = 'case:concept:name' if 'case:concept:name' in event_df.columns else None
ACTIVITY_COL = 'concept:name' if 'concept:name' in event_df.columns else None
RAW_ACTIVITY_COL = 'activity' if 'activity' in event_df.columns else ACTIVITY_COL
TIME_COL = 'time:timestamp' if 'time:timestamp' in event_df.columns else None
ORDER_COL = 'identity:id' if 'identity:id' in event_df.columns else 'eventid' if 'eventid' in event_df.columns else None
RESOURCE_COL = 'org:resource' if 'org:resource' in event_df.columns else None
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise RuntimeError(f'Kernspalten fehlen: CASE_COL={CASE_COL}, ACTIVITY_COL={ACTIVITY_COL}, TIME_COL={TIME_COL}')
event_df[CASE_COL] = event_df[CASE_COL].astype(str)
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce')
for c in ['doctype', 'subprocess', RAW_ACTIVITY_COL]:
    if c not in event_df.columns:
        event_df[c] = '__missing__'
event_df['combined_activity'] = event_df['doctype'].astype(str) + ' | ' + event_df['subprocess'].astype(str) + ' | ' + event_df[RAW_ACTIVITY_COL].astype(str)
event_df['_is_inspection_context'] = event_df['doctype'].astype(str).str.contains('inspection', case=False, na=False) | event_df['subprocess'].astype(str).str.contains('inspection|on-site|onsite', case=False, na=False, regex=True) | event_df['combined_activity'].astype(str).str.contains('inspection|on-site|onsite', case=False, na=False, regex=True)
sort_cols = [CASE_COL, TIME_COL]
if ORDER_COL is not None:
    sort_cols.append(ORDER_COL)
else:
    sort_cols.append(ACTIVITY_COL)
event_df = event_df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
event_df['_event_pos_in_case'] = event_df.groupby(CASE_COL).cumcount() + 1
event_df['_case_start'] = event_df.groupby(CASE_COL)[TIME_COL].transform('min')
event_df['_elapsed_days_from_case_start'] = (event_df[TIME_COL] - event_df['_case_start']).dt.total_seconds() / (3600 * 24)
basic_info = {'events': int(len(event_df)), 'cases': int(event_df[CASE_COL].nunique()), 'activities': int(event_df[ACTIVITY_COL].nunique()), 'raw_activities': int(event_df[RAW_ACTIVITY_COL].nunique()), 'combined_activities': int(event_df['combined_activity'].nunique()), 'columns': int(event_df.shape[1]), 'timestamp_min': str(event_df[TIME_COL].min()), 'timestamp_max': str(event_df[TIME_COL].max()), 'case_col': CASE_COL, 'activity_col': ACTIVITY_COL, 'raw_activity_col': RAW_ACTIVITY_COL, 'time_col': TIME_COL, 'order_col': ORDER_COL, 'resource_col': RESOURCE_COL}
save_json(basic_info, '00_basic_info_refinement.json')
print(json.dumps(basic_info, indent=2, ensure_ascii=False))


In [ ]:
# Fallmerkmale und Labels
case_times = event_df.groupby(CASE_COL)[TIME_COL].agg(case_start='min', case_end='max')
case_times['duration_days'] = (case_times['case_end'] - case_times['case_start']).dt.total_seconds() / (3600 * 24)
case_lengths = event_df.groupby(CASE_COL).size().rename('event_count')
case_df = case_times.join(case_lengths).reset_index()
case_df['case_start_month'] = case_df['case_start'].dt.month
case_df['case_start_quarter'] = case_df['case_start'].dt.quarter
case_df['case_start_weekday'] = case_df['case_start'].dt.weekday
case_df['case_start_year_from_timestamp'] = case_df['case_start'].dt.year
for col in ['case:year', 'case:department']:
    if col in event_df.columns:
        tmp = event_df.groupby(CASE_COL)[col].first().rename(col).reset_index()
        case_df = case_df.merge(tmp, on=CASE_COL, how='left')
first_event_cols = [CASE_COL, 'doctype', 'subprocess', RAW_ACTIVITY_COL, 'combined_activity']
if RESOURCE_COL is not None:
    first_event_cols.append(RESOURCE_COL)
first_events = event_df.groupby(CASE_COL, sort=False).first().reset_index()[first_event_cols].copy()
rename_map = {'doctype': 'first_doctype', 'subprocess': 'first_subprocess', RAW_ACTIVITY_COL: 'first_activity', 'combined_activity': 'first_combined_activity'}
if RESOURCE_COL is not None:
    rename_map[RESOURCE_COL] = 'first_resource'
first_events = first_events.rename(columns=rename_map)
case_df = case_df.merge(first_events, on=CASE_COL, how='left')
case_df = case_df.merge(event_df.groupby(CASE_COL)['doctype'].nunique().rename('n_doctypes_full_case').reset_index(), on=CASE_COL, how='left')
case_df = case_df.merge(event_df.groupby(CASE_COL)['subprocess'].nunique().rename('n_subprocesses_full_case').reset_index(), on=CASE_COL, how='left')
if RESOURCE_COL is not None:
    case_df = case_df.merge(event_df.groupby(CASE_COL)[RESOURCE_COL].nunique().rename('n_resources_full_case').reset_index(), on=CASE_COL, how='left')
else:
    case_df['n_resources_full_case'] = 0
combined_counts = event_df.groupby([CASE_COL, 'combined_activity']).size().rename('count').reset_index()
combined_rework = combined_counts.assign(extra=lambda x: np.maximum(x['count'] - 1, 0)).groupby(CASE_COL)['extra'].sum().rename('combined_rework_extra')
case_df = case_df.merge(combined_rework.reset_index(), on=CASE_COL, how='left')
case_df['combined_rework_extra'] = case_df['combined_rework_extra'].fillna(0).astype(int)
non_insp = event_df[~event_df['_is_inspection_context']].copy()
case_df = case_df.merge(non_insp.groupby(CASE_COL).size().rename('event_count_no_inspection').reset_index(), on=CASE_COL, how='left')
case_df['event_count_no_inspection'] = case_df['event_count_no_inspection'].fillna(0).astype(int)
non_insp_combined_counts = non_insp.groupby([CASE_COL, 'combined_activity']).size().rename('count').reset_index()
non_insp_combined_rework = non_insp_combined_counts.assign(extra=lambda x: np.maximum(x['count'] - 1, 0)).groupby(CASE_COL)['extra'].sum().rename('combined_rework_extra_no_inspection')
case_df = case_df.merge(non_insp_combined_rework.reset_index(), on=CASE_COL, how='left')
case_df['combined_rework_extra_no_inspection'] = case_df['combined_rework_extra_no_inspection'].fillna(0).astype(int)
case_df = case_df.merge(event_df.groupby(CASE_COL)['_is_inspection_context'].any().rename('has_inspection_context').reset_index(), on=CASE_COL, how='left')
case_df['has_inspection_context'] = robust_to_bool(case_df['has_inspection_context'])
CASE_YEAR_COL = 'case:year' if 'case:year' in case_df.columns else 'case_start_year_from_timestamp'
case_df[CASE_YEAR_COL] = case_df[CASE_YEAR_COL].astype(str)
thresholds = {'event_count_p90_global': quantile(case_df['event_count'], 0.9), 'combined_rework_p90_global': quantile(case_df['combined_rework_extra'], 0.9), 'event_count_p95_global': quantile(case_df['event_count'], 0.95), 'combined_rework_p95_global': quantile(case_df['combined_rework_extra'], 0.95), 'duration_p90_global': quantile(case_df['duration_days'], 0.9), 'duration_p95_global': quantile(case_df['duration_days'], 0.95), 'event_count_no_inspection_p90_global': quantile(case_df['event_count_no_inspection'], 0.9), 'combined_rework_no_inspection_p90_global': quantile(case_df['combined_rework_extra_no_inspection'], 0.9)}
case_df['label_scd_p90_or_global'] = (case_df['event_count'] >= thresholds['event_count_p90_global']) | (case_df['combined_rework_extra'] >= thresholds['combined_rework_p90_global'])
case_df['label_scd_p95_or_global'] = (case_df['event_count'] >= thresholds['event_count_p95_global']) | (case_df['combined_rework_extra'] >= thresholds['combined_rework_p95_global'])
case_df['label_scd_p90_and_global'] = (case_df['event_count'] >= thresholds['event_count_p90_global']) & (case_df['combined_rework_extra'] >= thresholds['combined_rework_p90_global'])
case_df['label_temporal_duration_p90_global'] = case_df['duration_days'] >= thresholds['duration_p90_global']
case_df['label_temporal_duration_p95_global'] = case_df['duration_days'] >= thresholds['duration_p95_global']
case_df['label_scd_p90_or_no_inspection_global'] = (case_df['event_count_no_inspection'] >= thresholds['event_count_no_inspection_p90_global']) | (case_df['combined_rework_extra_no_inspection'] >= thresholds['combined_rework_no_inspection_p90_global'])
for metric in ['event_count', 'combined_rework_extra', 'duration_days', 'event_count_no_inspection', 'combined_rework_extra_no_inspection']:
    for qname, qv in [('p90', 0.9), ('p95', 0.95)]:
        case_df[f'_thr_{metric}_{qname}_by_year'] = case_df.groupby(CASE_YEAR_COL)[metric].transform(lambda s: pd.to_numeric(s, errors='coerce').quantile(qv))
case_df['label_scd_p90_or_yearnorm'] = (case_df['event_count'] >= case_df['_thr_event_count_p90_by_year']) | (case_df['combined_rework_extra'] >= case_df['_thr_combined_rework_extra_p90_by_year'])
case_df['label_scd_p90_or_no_inspection_yearnorm'] = (case_df['event_count_no_inspection'] >= case_df['_thr_event_count_no_inspection_p90_by_year']) | (case_df['combined_rework_extra_no_inspection'] >= case_df['_thr_combined_rework_extra_no_inspection_p90_by_year'])
case_df['label_temporal_duration_p90_yearnorm'] = case_df['duration_days'] >= case_df['_thr_duration_days_p90_by_year']
by_case_sub = event_df.groupby([CASE_COL, 'subprocess']).size().unstack(fill_value=0) if 'subprocess' in event_df.columns else pd.DataFrame(index=case_df[CASE_COL])
change_like = [c for c in by_case_sub.columns if 'change' in str(c).lower()]
objection_like = [c for c in by_case_sub.columns if 'objection' in str(c).lower()]
if change_like or objection_like:
    path_label = by_case_sub[change_like + objection_like].sum(axis=1).gt(0).rename('label_path_change_or_objection').reset_index()
else:
    path_label = pd.DataFrame({CASE_COL: case_df[CASE_COL], 'label_path_change_or_objection': False})
case_df = case_df.merge(path_label, on=CASE_COL, how='left')
case_df['label_path_change_or_objection'] = robust_to_bool(case_df['label_path_change_or_objection'])
remove_label = event_df[RAW_ACTIVITY_COL].astype(str).str.lower().eq('remove document').groupby(event_df[CASE_COL]).any().rename('label_remove_document').reset_index()
case_df = case_df.merge(remove_label, on=CASE_COL, how='left')
case_df['label_remove_document'] = robust_to_bool(case_df['label_remove_document'])
LABEL_COLS = [PRIMARY_LABEL] + [c for c in ROBUSTNESS_LABELS if c in case_df.columns]
for col in set(LABEL_COLS + [c for c in case_df.columns if c.startswith('label_')]):
    if col in case_df.columns:
        case_df[col] = robust_to_bool(case_df[col])
label_diag = pd.DataFrame([{'label': col, 'positive_cases': int(case_df[col].sum()), 'share_positive_pct': float(case_df[col].mean() * 100)} for col in sorted([c for c in case_df.columns if c.startswith('label_')])])
save_csv(label_diag, '01_label_prevalence_refinement.csv', index=False)
save_json(thresholds, '01_label_thresholds_refinement.json')
save_csv(case_df.head(50), '01_case_df_preview_refinement_first50.csv', index=False)
display(label_diag.sort_values('share_positive_pct', ascending=False))
print('case_df:', case_df.shape)


In [ ]:
# Chronologischer Split
case_df = case_df.sort_values([CASE_YEAR_COL, 'case_start', CASE_COL], kind='mergesort').reset_index(drop=True)
year_values = sorted(case_df[CASE_YEAR_COL].dropna().astype(str).unique().tolist())
if set(['2015', '2016', '2017']).issubset(set(year_values)):
    case_df['split'] = np.select([case_df[CASE_YEAR_COL].astype(str).eq('2015'), case_df[CASE_YEAR_COL].astype(str).eq('2016'), case_df[CASE_YEAR_COL].astype(str).eq('2017')], ['train', 'validation', 'test'], default='unused')
    split_basis = 'case_year_2015_2016_2017'
else:
    case_df['_rank'] = np.arange(len(case_df))
    n = len(case_df)
    train_end = int(n * 1 / 3)
    val_end = int(n * 2 / 3)
    case_df['split'] = 'test'
    case_df.loc[:train_end - 1, 'split'] = 'train'
    case_df.loc[train_end:val_end - 1, 'split'] = 'validation'
    split_basis = 'case_start_time_quantiles'
case_df = case_df[case_df['split'].isin(['train', 'validation', 'test'])].copy()
split_summary = case_df.groupby('split').agg(n_cases=(CASE_COL, 'count'), positive_cases=(PRIMARY_LABEL, 'sum'), prevalence_pct=(PRIMARY_LABEL, lambda s: float(s.mean() * 100)), case_start_min=('case_start', 'min'), case_start_max=('case_start', 'max')).reset_index()
split_summary['positive_cases'] = split_summary['positive_cases'].astype(int)
save_csv(split_summary, '02_split_summary_primary_refinement.csv', index=False)
save_json({'split_basis': split_basis, 'case_year_col': CASE_YEAR_COL}, '02_split_basis_refinement.json')
display(split_summary)
FORBIDDEN_FEATURE_PATTERNS = ['duration_days', 'case_end', 'event_count', 'combined_rework_extra', 'full_case', 'label_', '_thr_', 'case_start', 'case_end', 'has_inspection_context', 'event_count_no_inspection', 'combined_rework_extra_no_inspection']
leakage_policy = pd.DataFrame([{'item': 'full duration', 'policy': 'exclude', 'reason': 'erst nach Fallende bekannt; Teil der temporalen Labels'}, {'item': 'full event_count', 'policy': 'exclude', 'reason': 'Bestandteil des SCD-Labels; direktes Leakage'}, {'item': 'full combined_rework_extra', 'policy': 'exclude', 'reason': 'Bestandteil des SCD-Labels; direktes Leakage'}, {'item': 'label_*', 'policy': 'exclude', 'reason': 'Target oder Target-Varianten'}, {'item': 'case:department', 'policy': 'scenario-dependent', 'reason': 'möglicherweise früh bekannt, aber als organisatorisches Signal kritisch'}, {'item': 'case:year', 'policy': 'scenario-dependent', 'reason': 'fachlich bekannt, aber mögliches Drift-/Jahresartefakt'}, {'item': 'resource features', 'policy': 'scenario-dependent', 'reason': 'operativ früh beobachtbar, aber kann organisatorische Struktur stark tragen'}, {'item': 'inspection features', 'policy': 'scenario-dependent', 'reason': 'fachlich relevant, aber kann SCD stark dominieren'}])
save_csv(leakage_policy, '03_feature_leakage_policy_refinement.csv', index=False)
leakage_policy


In [ ]:
# Präfixdatensätze
train_cases = set(case_df.loc[case_df['split'].eq('train'), CASE_COL])
all_case_index = case_df[[CASE_COL, 'split'] + LABEL_COLS].copy()

def select_prefix_events(prefix_spec):
    prefix_type = prefix_spec['prefix_type']
    value = prefix_spec['value']
    require_min_events = bool(prefix_spec.get('require_min_events', False))
    if prefix_type == 'event':
        sub = event_df[event_df['_event_pos_in_case'] <= int(value)].copy()
        if require_min_events:
            eligible = event_df.groupby(CASE_COL).size()
            eligible_cases = set(eligible[eligible >= int(value)].index.astype(str))
            sub = sub[sub[CASE_COL].isin(eligible_cases)].copy()
        return sub
    if prefix_type == 'time_days':
        sub = event_df[event_df['_elapsed_days_from_case_start'] <= float(value)].copy()
        return sub
    raise ValueError(f'Unbekannter prefix_type: {prefix_type}')

def compute_vocabularies(prefix_events):
    return {'raw_activities': top_values_from_train(prefix_events, RAW_ACTIVITY_COL, train_cases, TOP_N_RAW_ACTIVITIES), 'combined_activities': top_values_from_train(prefix_events, 'combined_activity', train_cases, TOP_N_COMBINED_ACTIVITIES), 'resources': top_values_from_train(prefix_events, RESOURCE_COL, train_cases, TOP_N_RESOURCES) if RESOURCE_COL is not None else []}

def add_count_features(base_df, prefix_events, col, values, prefix_name):
    if col is None or col not in prefix_events.columns or (not values):
        return base_df
    filtered = prefix_events[prefix_events[col].astype(str).isin([str(v) for v in values])]
    if len(filtered) == 0:
        for v in values:
            base_df[f'cnt_{prefix_name}__{sanitize_col_name(v)}'] = 0
        return base_df
    counts = filtered.groupby([CASE_COL, col]).size().unstack(fill_value=0)
    counts.columns = [f'cnt_{prefix_name}__{sanitize_col_name(c)}' for c in counts.columns]
    counts = counts.reset_index()
    base_df = base_df.merge(counts, on=CASE_COL, how='left')
    new_cols = [c for c in counts.columns if c != CASE_COL]
    base_df[new_cols] = base_df[new_cols].fillna(0).astype(int)
    return base_df

def repeated_extra(prefix_events, col, out_name):
    if col not in prefix_events.columns or len(prefix_events) == 0:
        return pd.DataFrame({CASE_COL: [], out_name: []})
    tmp = prefix_events.groupby([CASE_COL, col]).size().rename('count').reset_index()
    tmp['extra'] = np.maximum(tmp['count'] - 1, 0)
    return tmp.groupby(CASE_COL)['extra'].sum().rename(out_name).reset_index()

def build_prefix_dataset(prefix_spec):
    prefix_id = prefix_spec['prefix_id']
    prefix_events = select_prefix_events(prefix_spec)
    if prefix_spec.get('require_min_events', False):
        eligible_cases = set(prefix_events[CASE_COL].unique())
        base = case_df[case_df[CASE_COL].isin(eligible_cases)].copy()
    else:
        base = case_df.copy()
    feature_df = base[[CASE_COL, 'split'] + LABEL_COLS].copy()
    static_cols = ['case_start_month', 'case_start_quarter', 'case_start_weekday', 'case_start_year_from_timestamp', 'case:year', 'case:department', 'first_doctype', 'first_subprocess', 'first_activity', 'first_combined_activity', 'first_resource']
    for col in static_cols:
        if col in base.columns:
            feature_df[col] = base[col]
    counts = prefix_events.groupby(CASE_COL).size().rename('prefix_event_count').reset_index()
    feature_df = feature_df.merge(counts, on=CASE_COL, how='left')
    feature_df['prefix_event_count'] = feature_df['prefix_event_count'].fillna(0).astype(int)
    if len(prefix_events):
        prefix_time = prefix_events.groupby(CASE_COL)[TIME_COL].agg(prefix_start='min', prefix_last='max').reset_index()
        prefix_time['prefix_duration_hours'] = (prefix_time['prefix_last'] - prefix_time['prefix_start']).dt.total_seconds() / 3600
        feature_df = feature_df.merge(prefix_time[[CASE_COL, 'prefix_duration_hours']], on=CASE_COL, how='left')
    else:
        feature_df['prefix_duration_hours'] = 0
    feature_df['prefix_duration_hours'] = feature_df['prefix_duration_hours'].fillna(0)
    for col, out_name in [(RAW_ACTIVITY_COL, 'prefix_n_raw_activities'), ('combined_activity', 'prefix_n_combined_activities'), ('doctype', 'prefix_n_doctypes'), ('subprocess', 'prefix_n_subprocesses')]:
        if col in prefix_events.columns and len(prefix_events):
            tmp = prefix_events.groupby(CASE_COL)[col].nunique().rename(out_name).reset_index()
            feature_df = feature_df.merge(tmp, on=CASE_COL, how='left')
        else:
            feature_df[out_name] = 0
        feature_df[out_name] = feature_df[out_name].fillna(0).astype(int)
    if RESOURCE_COL is not None and RESOURCE_COL in prefix_events.columns and len(prefix_events):
        tmp = prefix_events.groupby(CASE_COL)[RESOURCE_COL].nunique().rename('prefix_n_resources').reset_index()
        feature_df = feature_df.merge(tmp, on=CASE_COL, how='left')
    else:
        feature_df['prefix_n_resources'] = 0
    feature_df['prefix_n_resources'] = feature_df['prefix_n_resources'].fillna(0).astype(int)
    feature_df = feature_df.merge(repeated_extra(prefix_events, RAW_ACTIVITY_COL, 'prefix_raw_rework_extra'), on=CASE_COL, how='left')
    feature_df = feature_df.merge(repeated_extra(prefix_events, 'combined_activity', 'prefix_combined_rework_extra'), on=CASE_COL, how='left')
    feature_df['prefix_raw_rework_extra'] = feature_df['prefix_raw_rework_extra'].fillna(0).astype(int)
    feature_df['prefix_combined_rework_extra'] = feature_df['prefix_combined_rework_extra'].fillna(0).astype(int)

    def any_by_case(mask, name):
        if len(prefix_events) == 0:
            feature_df[name] = False
            return
        tmp = mask.groupby(prefix_events[CASE_COL]).any().rename(name).reset_index()
        nonlocal_feature[0] = nonlocal_feature[0].merge(tmp, on=CASE_COL, how='left')
        nonlocal_feature[0][name] = robust_to_bool(nonlocal_feature[0][name])
    nonlocal_feature = [feature_df]
    if 'subprocess' in prefix_events.columns and len(prefix_events):
        any_by_case(prefix_events['subprocess'].astype(str).str.contains('change|objection', case=False, na=False, regex=True), 'prefix_has_change_or_objection')
    else:
        nonlocal_feature[0]['prefix_has_change_or_objection'] = False
    if len(prefix_events):
        any_by_case(prefix_events['_is_inspection_context'].fillna(False).astype(bool), 'prefix_has_inspection_context')
        any_by_case(prefix_events[RAW_ACTIVITY_COL].astype(str).str.lower().eq('remove document'), 'prefix_has_remove_document')
        any_by_case(prefix_events[RAW_ACTIVITY_COL].astype(str).str.contains('payment', case=False, na=False), 'prefix_has_payment_activity')
        any_by_case(prefix_events[RAW_ACTIVITY_COL].astype(str).str.contains('decide|decision', case=False, na=False, regex=True), 'prefix_has_decision_activity')
    else:
        for c in ['prefix_has_inspection_context', 'prefix_has_remove_document', 'prefix_has_payment_activity', 'prefix_has_decision_activity']:
            nonlocal_feature[0][c] = False
    feature_df = nonlocal_feature[0]
    if len(prefix_events):
        last_cols = [CASE_COL, RAW_ACTIVITY_COL, 'subprocess', 'doctype']
        if RESOURCE_COL is not None:
            last_cols.append(RESOURCE_COL)
        last_events = prefix_events.groupby(CASE_COL, sort=False).last().reset_index()[last_cols].copy()
        rename_last = {RAW_ACTIVITY_COL: 'last_activity_prefix', 'subprocess': 'last_subprocess_prefix', 'doctype': 'last_doctype_prefix'}
        if RESOURCE_COL is not None:
            rename_last[RESOURCE_COL] = 'last_resource_prefix'
        last_events = last_events.rename(columns=rename_last)
        feature_df = feature_df.merge(last_events, on=CASE_COL, how='left')
    for c in ['last_activity_prefix', 'last_subprocess_prefix', 'last_doctype_prefix', 'last_resource_prefix']:
        if c not in feature_df.columns:
            feature_df[c] = '__none__'
        feature_df[c] = feature_df[c].fillna('__none__').astype(str)
    vocab = compute_vocabularies(prefix_events)
    feature_df = add_count_features(feature_df, prefix_events, RAW_ACTIVITY_COL, vocab['raw_activities'], 'raw_activity')
    feature_df = add_count_features(feature_df, prefix_events, 'combined_activity', vocab['combined_activities'], 'combined')
    if RESOURCE_COL is not None:
        feature_df = add_count_features(feature_df, prefix_events, RESOURCE_COL, vocab['resources'], 'resource')
    feature_df['_prefix_id'] = prefix_id
    feature_df['_prefix_type'] = prefix_spec['prefix_type']
    feature_df['_prefix_value'] = prefix_spec['value']
    for col in feature_df.select_dtypes(include=['object', 'string', 'category']).columns:
        if col not in [CASE_COL, 'split'] and (not col.startswith('label_')):
            feature_df[col] = feature_df[col].fillna('__missing__').astype(str)
    for col in feature_df.columns:
        if col.startswith('prefix_has_') or feature_df[col].dtype == bool:
            if col not in LABEL_COLS:
                feature_df[col] = robust_to_bool(feature_df[col]).astype(int)
    schema = {'prefix_id': prefix_id, 'prefix_type': prefix_spec['prefix_type'], 'prefix_value': prefix_spec['value'], 'require_min_events': bool(prefix_spec.get('require_min_events', False)), 'n_rows': int(len(feature_df)), 'mean_prefix_event_count': float(feature_df['prefix_event_count'].mean()), 'median_prefix_event_count': float(feature_df['prefix_event_count'].median()), 'share_zero_prefix_events_pct': float((feature_df['prefix_event_count'] == 0).mean() * 100), 'vocab_raw_activities': len(vocab['raw_activities']), 'vocab_combined_activities': len(vocab['combined_activities']), 'vocab_resources': len(vocab['resources'])}
    return (feature_df, schema, vocab)
prefix_datasets = {}
prefix_schemas = []
prefix_vocabs = {}
for spec in PRIMARY_PREFIX_SPECS:
    print('Baue Prefix-Dataset:', spec['prefix_id'])
    ds, schema, vocab = build_prefix_dataset(spec)
    prefix_datasets[spec['prefix_id']] = ds
    prefix_schemas.append(schema)
    prefix_vocabs[spec['prefix_id']] = vocab
    save_csv(ds.head(50), f"04_modeling_dataset_preview_{spec['prefix_id']}.csv", index=False)
prefix_schema_df = pd.DataFrame(prefix_schemas)
save_csv(prefix_schema_df, '04_prefix_dataset_schema_summary.csv', index=False)
save_json(prefix_vocabs, '04_prefix_feature_vocabularies_train_only.json')
display(prefix_schema_df)


In [ ]:
# Ablation und Modelle
ID_AND_META_COLS = {CASE_COL, 'split', '_prefix_id', '_prefix_type', '_prefix_value'}

def is_forbidden_feature(col):
    c = str(col).lower()
    if col in ID_AND_META_COLS:
        return True
    if c.startswith('label_'):
        return True
    if c.startswith('_thr_'):
        return True
    exact_forbidden = {'duration_days', 'case_start', 'case_end', 'event_count', 'combined_rework_extra', 'event_count_no_inspection', 'combined_rework_extra_no_inspection', 'has_inspection_context'}
    if c in exact_forbidden:
        return True
    if 'full_case' in c:
        return True
    return False

def base_feature_columns(df, target_col):
    cols = []
    for col in df.columns:
        if col == target_col:
            continue
        if is_forbidden_feature(col):
            continue
        cols.append(col)
    return cols

def apply_ablation_scenario(feature_cols, scenario_name):
    cfg = ABLATION_SCENARIOS[scenario_name]
    cols = list(feature_cols)
    if cfg.get('keep_only_prefix_features', False):
        keep_prefix_patterns = ['prefix_', 'cnt_', 'last_']
        cols = [c for c in cols if any((str(c).lower().startswith(p) for p in keep_prefix_patterns))]
    for pat in cfg.get('drop_patterns', []):
        p = pat.lower()
        cols = [c for c in cols if p not in str(c).lower()]
    return cols

def infer_feature_types(df, feature_cols):
    numeric_cols = []
    categorical_cols = []
    diagnostics = []
    for col in feature_cols:
        s = df[col]
        col_lower = str(col).lower()
        force_cat = any((x in col_lower for x in ['case:department', 'department', 'doctype', 'subprocess', 'activity', 'combined_activity', 'resource', 'first_', 'last_']))
        if force_cat:
            categorical_cols.append(col)
            diagnostics.append({'feature': col, 'assigned_type': 'categorical', 'reason': 'forced_process_context'})
            continue
        if pd.api.types.is_bool_dtype(s):
            numeric_cols.append(col)
            diagnostics.append({'feature': col, 'assigned_type': 'numeric', 'reason': 'bool'})
            continue
        if pd.api.types.is_numeric_dtype(s):
            numeric_cols.append(col)
            diagnostics.append({'feature': col, 'assigned_type': 'numeric', 'reason': 'numeric_dtype'})
            continue
        converted = pd.to_numeric(s, errors='coerce')
        convertible_share = converted.notna().mean()
        if convertible_share >= 0.98:
            numeric_cols.append(col)
            diagnostics.append({'feature': col, 'assigned_type': 'numeric', 'reason': f'numeric_like_{convertible_share:.2f}'})
        else:
            categorical_cols.append(col)
            diagnostics.append({'feature': col, 'assigned_type': 'categorical', 'reason': f'not_numeric_like_{convertible_share:.2f}'})
    return (numeric_cols, categorical_cols, pd.DataFrame(diagnostics))

def make_pipeline(model_name, numeric_cols, categorical_cols):
    transformers = []
    if numeric_cols:
        transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols))
    if categorical_cols:
        transformers.append(('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', make_ohe())]), categorical_cols))
    if not transformers and model_name != 'dummy_prior':
        raise RuntimeError('Keine Features nach Ablation vorhanden.')
    preprocess = ColumnTransformer(transformers=transformers, remainder='drop')
    if model_name == 'dummy_prior':
        model = DummyClassifier(strategy='prior')
    elif model_name == 'logreg_balanced':
        model = LogisticRegression(max_iter=2500, class_weight='balanced', solver='lbfgs', random_state=RANDOM_STATE)
    elif model_name == 'rf_balanced':
        model = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, min_samples_leaf=RF_MIN_SAMPLES_LEAF, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)
    else:
        raise ValueError(f'Unbekanntes Modell: {model_name}')
    return Pipeline([('preprocess', preprocess), ('model', model)])

def split_X_y(df, feature_cols, target_col):
    parts = {}
    for sp in ['train', 'validation', 'test']:
        sub = df[df['split'].eq(sp)].copy()
        X = sub[feature_cols].copy()
        y = robust_to_bool(sub[target_col]).astype(int).values
        parts[sp] = {'X': X, 'y': y, 'df': sub[[CASE_COL, 'split', target_col]].copy()}
    return parts

def get_scores(pipe, X):
    if hasattr(pipe.named_steps['model'], 'predict_proba'):
        proba = pipe.predict_proba(X)
        if proba.shape[1] == 1:
            return np.repeat(float(proba[:, 0][0]), len(X))
        return proba[:, 1]
    if hasattr(pipe.named_steps['model'], 'decision_function'):
        scores = pipe.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    return pipe.predict(X).astype(float)

def fit_evaluate_dataset(df, target_col, prefix_id, scenario_name, model_name):
    if target_col not in df.columns:
        raise RuntimeError(f'Target fehlt: {target_col}')
    df = df.copy()
    df[target_col] = robust_to_bool(df[target_col])
    if df[df['split'].eq('train')][target_col].nunique() < 2:
        raise RuntimeError(f'Train split enthält nur eine Klasse für {target_col}')
    base_cols = base_feature_columns(df, target_col)
    feature_cols = apply_ablation_scenario(base_cols, scenario_name)
    if len(feature_cols) == 0 and model_name != 'dummy_prior':
        raise RuntimeError(f'Keine Features nach Scenario {scenario_name}')
    numeric_cols, categorical_cols, feature_diag = infer_feature_types(df, feature_cols)
    parts = split_X_y(df, feature_cols, target_col)
    pipe = make_pipeline(model_name, numeric_cols, categorical_cols)
    pipe.fit(parts['train']['X'], parts['train']['y'])
    scores = {sp: get_scores(pipe, parts[sp]['X']) for sp in ['train', 'validation', 'test']}
    val_thr, val_best_f1 = select_threshold_by_f1(parts['validation']['y'], scores['validation'])
    eval_rows = []
    for sp in ['train', 'validation', 'test']:
        eval_rows.append(evaluate_scores(parts[sp]['y'], scores[sp], 0.5, target_col, prefix_id, scenario_name, model_name, sp, 'fixed_0_5'))
        eval_rows.append(evaluate_scores(parts[sp]['y'], scores[sp], val_thr, target_col, prefix_id, scenario_name, model_name, sp, 'validation_f1_threshold'))
    eval_df = pd.DataFrame(eval_rows)
    eval_df['validation_selected_threshold'] = val_thr
    eval_df['validation_best_f1'] = val_best_f1
    eval_df['raw_feature_count'] = len(feature_cols)
    eval_df['numeric_feature_count'] = len(numeric_cols)
    eval_df['categorical_feature_count'] = len(categorical_cols)
    pred_test = parts['test']['df'].copy()
    pred_test['target'] = target_col
    pred_test['prefix_id'] = prefix_id
    pred_test['scenario'] = scenario_name
    pred_test['model'] = model_name
    pred_test['y_true'] = parts['test']['y']
    pred_test['score'] = scores['test']
    pred_test['pred_fixed_0_5'] = (scores['test'] >= 0.5).astype(int)
    pred_test['pred_validation_f1_threshold'] = (scores['test'] >= val_thr).astype(int)
    fi_df = pd.DataFrame()
    feature_names = get_feature_names_from_pipeline(pipe)
    model = pipe.named_steps['model']
    try:
        if model_name == 'logreg_balanced' and hasattr(model, 'coef_'):
            fi_df = pd.DataFrame({'feature': feature_names, 'importance': model.coef_[0]})
        elif model_name == 'rf_balanced' and hasattr(model, 'feature_importances_'):
            fi_df = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_})
        if len(fi_df):
            fi_df['target'] = target_col
            fi_df['prefix_id'] = prefix_id
            fi_df['scenario'] = scenario_name
            fi_df['model'] = model_name
            fi_df['abs_importance'] = fi_df['importance'].abs()
    except Exception as e:
        analysis_notes.append(f'Feature importance failed: {target_col}/{prefix_id}/{scenario_name}/{model_name}: {repr(e)}')
    calib_rows = []
    y_test = parts['test']['y']
    s_test = scores['test']
    try:
        bins = pd.qcut(pd.Series(s_test), q=10, duplicates='drop')
        calib = pd.DataFrame({'score': s_test, 'y': y_test, 'bin': bins})
        calib_rows = calib.groupby('bin', observed=True).agg(mean_predicted_score=('score', 'mean'), observed_positive_rate=('y', 'mean'), n=('y', 'count')).reset_index(drop=True)
        calib_rows['target'] = target_col
        calib_rows['prefix_id'] = prefix_id
        calib_rows['scenario'] = scenario_name
        calib_rows['model'] = model_name
    except Exception:
        calib_rows = pd.DataFrame()
    curves = []
    try:
        if len(np.unique(y_test)) >= 2:
            fpr, tpr, _ = roc_curve(y_test, s_test)
            curves.append(pd.DataFrame({'x': fpr, 'y': tpr, 'curve': 'roc', 'target': target_col, 'prefix_id': prefix_id, 'scenario': scenario_name, 'model': model_name}))
            prec, rec, _ = precision_recall_curve(y_test, s_test)
            curves.append(pd.DataFrame({'x': rec, 'y': prec, 'curve': 'pr', 'target': target_col, 'prefix_id': prefix_id, 'scenario': scenario_name, 'model': model_name}))
    except Exception:
        pass
    curve_df = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()
    feature_diag['target'] = target_col
    feature_diag['prefix_id'] = prefix_id
    feature_diag['scenario'] = scenario_name
    feature_diag['model'] = model_name
    return {'eval_df': eval_df, 'pred_test_df': pred_test, 'feature_importance_df': fi_df, 'calibration_df': calib_rows, 'curve_df': curve_df, 'feature_diag_df': feature_diag}
print('Modeling-Funktionen geladen.')


In [ ]:
# Hauptlabel verfeinern
primary_eval_parts = []
primary_pred_parts = []
primary_fi_parts = []
primary_calib_parts = []
primary_curve_parts = []
primary_feature_diag_parts = []
failure_rows = []
for spec in PRIMARY_PREFIX_SPECS:
    prefix_id = spec['prefix_id']
    df_prefix = prefix_datasets[prefix_id]
    print('\n' + '=' * 90)
    print('Primary Target:', PRIMARY_LABEL, '| Prefix:', prefix_id, '| rows:', len(df_prefix))
    for scenario_name in ABLATION_SCENARIOS.keys():
        for model_name in MODELS_MAIN:
            print(f'Trainiere: prefix={prefix_id}, scenario={scenario_name}, model={model_name}')
            try:
                res = fit_evaluate_dataset(df_prefix, PRIMARY_LABEL, prefix_id, scenario_name, model_name)
                primary_eval_parts.append(res['eval_df'])
                primary_pred_parts.append(res['pred_test_df'])
                if len(res['feature_importance_df']):
                    primary_fi_parts.append(res['feature_importance_df'].sort_values('abs_importance', ascending=False).head(150))
                if len(res['calibration_df']):
                    primary_calib_parts.append(res['calibration_df'])
                if len(res['curve_df']):
                    primary_curve_parts.append(res['curve_df'])
                primary_feature_diag_parts.append(res['feature_diag_df'])
            except Exception as e:
                msg = repr(e)
                print('  FAILED:', msg)
                failure_rows.append({'stage': 'primary_refinement', 'target': PRIMARY_LABEL, 'prefix_id': prefix_id, 'scenario': scenario_name, 'model': model_name, 'error': msg})
primary_refinement_eval_df = pd.concat(primary_eval_parts, ignore_index=True) if primary_eval_parts else pd.DataFrame()
primary_refinement_predictions_df = pd.concat(primary_pred_parts, ignore_index=True) if primary_pred_parts else pd.DataFrame()
primary_refinement_fi_df = pd.concat(primary_fi_parts, ignore_index=True) if primary_fi_parts else pd.DataFrame()
primary_refinement_calibration_df = pd.concat(primary_calib_parts, ignore_index=True) if primary_calib_parts else pd.DataFrame()
primary_refinement_curve_df = pd.concat(primary_curve_parts, ignore_index=True) if primary_curve_parts else pd.DataFrame()
primary_refinement_feature_diag_df = pd.concat(primary_feature_diag_parts, ignore_index=True) if primary_feature_diag_parts else pd.DataFrame()
failure_log_df = pd.DataFrame(failure_rows)
save_csv(primary_refinement_eval_df, '05_primary_refinement_evaluation_all_splits.csv', index=False)
save_csv(primary_refinement_predictions_df, '06_primary_refinement_test_predictions.csv', index=False)
save_csv(primary_refinement_fi_df, '07_primary_refinement_feature_importances_top150.csv', index=False)
save_csv(primary_refinement_calibration_df, '08_primary_refinement_calibration_bins.csv', index=False)
save_csv(primary_refinement_curve_df, '09_primary_refinement_roc_pr_curves.csv', index=False)
save_csv(primary_refinement_feature_diag_df, '10_primary_refinement_feature_type_diagnostics.csv', index=False)
save_csv(failure_log_df, '10b_primary_refinement_failure_log.csv', index=False)
print('Primary refinement evaluation rows:', len(primary_refinement_eval_df))
print('Failures:', len(failure_log_df))
if len(primary_refinement_eval_df) == 0:
    display(failure_log_df)
    raise RuntimeError('Keine Primary-Refinement-Ergebnisse erzeugt. Failure Log prüfen.')
display(primary_refinement_eval_df.head())


In [ ]:
# Robustheitslabels
robust_parts = []
robust_fi_parts = []
robust_failure_rows = []
available_robust_targets = [t for t in ROBUSTNESS_LABELS if t in case_df.columns]
print('Robustheitslabels verfügbar:', available_robust_targets)
for spec in ROBUSTNESS_PREFIX_SPECS:
    prefix_id = spec['prefix_id']
    if prefix_id not in prefix_datasets:
        ds, schema, vocab = build_prefix_dataset(spec)
        prefix_datasets[prefix_id] = ds
    df_prefix = prefix_datasets[prefix_id]
    for target in available_robust_targets:
        if df_prefix[target].nunique() < 2:
            robust_failure_rows.append({'stage': 'robustness', 'target': target, 'prefix_id': prefix_id, 'scenario': 'full_allowed', 'model': None, 'error': 'target has one class'})
            continue
        for model_name in ROBUSTNESS_MODELS:
            print(f'Robustheit: target={target}, prefix={prefix_id}, model={model_name}')
            try:
                res = fit_evaluate_dataset(df_prefix, target, prefix_id, 'full_allowed', model_name)
                robust_parts.append(res['eval_df'])
                if len(res['feature_importance_df']):
                    robust_fi_parts.append(res['feature_importance_df'].sort_values('abs_importance', ascending=False).head(100))
            except Exception as e:
                robust_failure_rows.append({'stage': 'robustness', 'target': target, 'prefix_id': prefix_id, 'scenario': 'full_allowed', 'model': model_name, 'error': repr(e)})
robustness_refinement_eval_df = pd.concat(robust_parts, ignore_index=True) if robust_parts else pd.DataFrame()
robustness_refinement_fi_df = pd.concat(robust_fi_parts, ignore_index=True) if robust_fi_parts else pd.DataFrame()
robustness_failure_log_df = pd.DataFrame(robust_failure_rows)
save_csv(robustness_refinement_eval_df, '11_robustness_refinement_evaluation.csv', index=False)
save_csv(robustness_refinement_fi_df, '12_robustness_refinement_feature_importances_top100.csv', index=False)
save_csv(robustness_failure_log_df, '12b_robustness_refinement_failure_log.csv', index=False)
print('Robustness rows:', len(robustness_refinement_eval_df))
print('Robustness failures:', len(robustness_failure_log_df))
if len(robustness_refinement_eval_df):
    display(robustness_refinement_eval_df.head())
else:
    display(robustness_failure_log_df)


In [ ]:
# Modellvergleich
primary_test_fixed = primary_refinement_eval_df[primary_refinement_eval_df['split'].eq('test') & primary_refinement_eval_df['threshold_policy'].eq('fixed_0_5')].copy()
primary_test_selected = primary_refinement_eval_df[primary_refinement_eval_df['split'].eq('test') & primary_refinement_eval_df['threshold_policy'].eq('validation_f1_threshold')].copy()
primary_test_fixed_rank = primary_test_fixed.sort_values('pr_auc_average_precision', ascending=False)
primary_test_selected_rank = primary_test_selected.sort_values('f1', ascending=False)
save_csv(primary_test_fixed_rank, '13_primary_refinement_test_ranking_fixed_threshold.csv', index=False)
save_csv(primary_test_selected_rank, '14_primary_refinement_test_ranking_validation_threshold.csv', index=False)
best_primary_by_pr = primary_test_fixed_rank.iloc[0].to_dict() if len(primary_test_fixed_rank) else {}
save_json(best_primary_by_pr, '15_best_refinement_model_by_test_pr_auc.json')
full_ref = primary_test_fixed[primary_test_fixed['scenario'].eq('full_allowed')][['prefix_id', 'model', 'pr_auc_average_precision', 'roc_auc', 'f1']].rename(columns={'pr_auc_average_precision': 'full_pr_auc', 'roc_auc': 'full_roc_auc', 'f1': 'full_f1'})
ablation_impact = primary_test_fixed.merge(full_ref, on=['prefix_id', 'model'], how='left')
ablation_impact['delta_pr_auc_vs_full'] = ablation_impact['pr_auc_average_precision'] - ablation_impact['full_pr_auc']
ablation_impact['delta_roc_auc_vs_full'] = ablation_impact['roc_auc'] - ablation_impact['full_roc_auc']
ablation_impact['delta_f1_vs_full'] = ablation_impact['f1'] - ablation_impact['full_f1']
save_csv(ablation_impact, '16_ablation_impact_vs_full_allowed.csv', index=False)
prefix_comparison = primary_test_fixed[primary_test_fixed['scenario'].eq('full_allowed')].copy()
save_csv(prefix_comparison.sort_values(['model', 'pr_auc_average_precision'], ascending=[True, False]), '17_prefix_comparison_full_allowed.csv', index=False)
if len(robustness_refinement_eval_df):
    robustness_test_rank = robustness_refinement_eval_df[robustness_refinement_eval_df['split'].eq('test') & robustness_refinement_eval_df['threshold_policy'].eq('fixed_0_5')].sort_values(['target', 'pr_auc_average_precision'], ascending=[True, False])
    save_csv(robustness_test_rank, '18_robustness_refinement_test_ranking.csv', index=False)
else:
    robustness_test_rank = pd.DataFrame()
baseline05_path = BASELINE_05_TABLE_DIR / '07_primary_model_evaluation_all_splits.csv'
if baseline05_path.exists():
    baseline05 = pd.read_csv(baseline05_path)
    baseline05_test = baseline05[baseline05['split'].eq('test') & baseline05['threshold_policy'].eq('fixed_0_5')].copy()
    baseline05_test['source'] = '05_baseline'
    current_comp = primary_test_fixed.copy()
    current_comp['source'] = '06_refinement'
    keep = ['source', 'target', 'prefix_id', 'model', 'scenario', 'pr_auc_average_precision', 'roc_auc', 'f1', 'precision', 'recall', 'balanced_accuracy', 'brier_score']
    if 'prefix_len' in baseline05_test.columns:
        baseline05_test['prefix_id'] = 'event' + baseline05_test['prefix_len'].astype(str)
    if 'scenario' not in baseline05_test.columns:
        baseline05_test['scenario'] = 'baseline05_full'
    baseline_compare = pd.concat([baseline05_test[[c for c in keep if c in baseline05_test.columns]], current_comp[[c for c in keep if c in current_comp.columns]]], ignore_index=True)
    save_csv(baseline_compare, '19_comparison_with_notebook05_baseline.csv', index=False)
else:
    baseline_compare = pd.DataFrame()
    analysis_notes.append('Notebook-05-Baseline-Datei nicht gefunden; Vergleich wird übersprungen.')
print('Best refinement model by Test PR-AUC:')
print(best_primary_by_pr)
print('\nTop 10 Primary Test Ranking:')
display(primary_test_fixed_rank.head(10))


In [ ]:
# Abbildungen
fig, ax = plt.subplots(figsize=(11, 5))
plot_df = primary_test_fixed[primary_test_fixed['scenario'].eq('full_allowed')].copy()
for model in plot_df['model'].unique():
    sub = plot_df[plot_df['model'].eq(model)].copy()
    order = {p['prefix_id']: i for i, p in enumerate(PRIMARY_PREFIX_SPECS)}
    sub['_order'] = sub['prefix_id'].map(order)
    sub = sub.sort_values('_order')
    ax.plot(sub['prefix_id'], sub['pr_auc_average_precision'], marker='o', label=model)
ax.set_xlabel('Prefix')
ax.set_ylabel('Test PR-AUC / Average Precision')
ax.set_title('Full-Allowed: PR-AUC nach Prefix')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30, ha='right')
save_fig(fig, 'fig_01_pr_auc_by_prefix_full_allowed.png')
fig, ax = plt.subplots(figsize=(11, 6))
impact_plot = ablation_impact[ablation_impact['prefix_id'].eq('event20_all') & ablation_impact['model'].isin(['logreg_balanced', 'rf_balanced']) & ~ablation_impact['scenario'].eq('full_allowed')].copy()
if len(impact_plot):
    for model in impact_plot['model'].unique():
        sub = impact_plot[impact_plot['model'].eq(model)].sort_values('delta_pr_auc_vs_full')
        ax.plot(sub['delta_pr_auc_vs_full'], sub['scenario'], marker='o', label=model)
    ax.axvline(0, linestyle='--')
    ax.set_xlabel('Delta Test PR-AUC gegenüber full_allowed')
    ax.set_ylabel('Ablation Scenario')
    ax.set_title('Ablation Impact bei Event-Prefix 20')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)
    save_fig(fig, 'fig_02_ablation_impact_event20.png')
else:
    plt.close(fig)
fig, ax = plt.subplots(figsize=(9, 5))
time_order = {'time7d': 7, 'time30d': 30, 'time60d': 60, 'time90d': 90}
time_plot = plot_df[plot_df['prefix_id'].isin(time_order.keys())].copy()
time_plot['days'] = time_plot['prefix_id'].map(time_order)
for model in time_plot['model'].unique():
    sub = time_plot[time_plot['model'].eq(model)].sort_values('days')
    ax.plot(sub['days'], sub['pr_auc_average_precision'], marker='o', label=model)
ax.set_xlabel('Tage seit Case-Start')
ax.set_ylabel('Test PR-AUC')
ax.set_title('Time-Based Prefixes: PR-AUC nach Beobachtungsfenster')
ax.legend()
ax.grid(alpha=0.3)
save_fig(fig, 'fig_03_time_prefix_pr_auc.png')
fig, ax = plt.subplots(figsize=(9, 5))
time_schema = prefix_schema_df[prefix_schema_df['prefix_type'].eq('time_days')].copy().sort_values('prefix_value')
ax.plot(time_schema['prefix_value'], time_schema['median_prefix_event_count'], marker='o', label='Median Events')
ax.plot(time_schema['prefix_value'], time_schema['mean_prefix_event_count'], marker='o', label='Mean Events')
ax.set_xlabel('Tage seit Case-Start')
ax.set_ylabel('Events im Prefix')
ax.set_title('Wie viel Information enthalten Time-Based Prefixes?')
ax.legend()
ax.grid(alpha=0.3)
save_fig(fig, 'fig_04_time_prefix_information_volume.png')
if len(robustness_test_rank):
    fig, ax = plt.subplots(figsize=(11, 6))
    rob_plot = robustness_test_rank[robustness_test_rank['model'].isin(ROBUSTNESS_MODELS)].copy()
    for model in rob_plot['model'].unique():
        sub = rob_plot[rob_plot['model'].eq(model)].copy()
        sub['label'] = sub['target'] + ' | ' + sub['prefix_id']
        sub = sub.sort_values('pr_auc_average_precision')
        ax.scatter(sub['pr_auc_average_precision'], sub['label'], label=model, alpha=0.8)
    ax.set_xlabel('Test PR-AUC')
    ax.set_title('Robustheitslabels: PR-AUC unter ausgewählten Prefixes')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)
    save_fig(fig, 'fig_05_robustness_targets_pr_auc.png')
if len(best_primary_by_pr) and len(primary_refinement_calibration_df):
    best_model = best_primary_by_pr.get('model')
    best_prefix = best_primary_by_pr.get('prefix_id')
    best_scenario = best_primary_by_pr.get('scenario')
    calib = primary_refinement_calibration_df[primary_refinement_calibration_df['model'].eq(best_model) & primary_refinement_calibration_df['prefix_id'].eq(best_prefix) & primary_refinement_calibration_df['scenario'].eq(best_scenario)].copy()
    if len(calib):
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot(calib['mean_predicted_score'], calib['observed_positive_rate'], marker='o')
        ax.plot([0, 1], [0, 1], linestyle='--')
        ax.set_xlabel('Mittlerer vorhergesagter Score')
        ax.set_ylabel('Beobachtete positive Rate')
        ax.set_title(f'Calibration: {best_model}, {best_prefix}, {best_scenario}')
        ax.grid(alpha=0.3)
        save_fig(fig, 'fig_06_calibration_best_refinement_model.png')
logreg_rows = primary_test_fixed[primary_test_fixed['model'].eq('logreg_balanced') & primary_test_fixed['scenario'].eq('full_allowed')].sort_values('pr_auc_average_precision', ascending=False)
if len(logreg_rows) and len(primary_refinement_fi_df):
    row = logreg_rows.iloc[0]
    fi = primary_refinement_fi_df[primary_refinement_fi_df['model'].eq('logreg_balanced') & primary_refinement_fi_df['prefix_id'].eq(row['prefix_id']) & primary_refinement_fi_df['scenario'].eq('full_allowed')].copy()
    if len(fi):
        top_pos = fi.sort_values('importance', ascending=False).head(12)
        top_neg = fi.sort_values('importance', ascending=True).head(12)
        top_combined = pd.concat([top_neg, top_pos], ignore_index=True).sort_values('importance')
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(top_combined['feature'], top_combined['importance'])
        ax.set_xlabel('Logistic Regression Koeffizient')
        ax.set_title(f"Top LogReg Features: {row['prefix_id']}")
        ax.grid(axis='x', alpha=0.3)
        save_fig(fig, 'fig_07_top_logreg_features_refinement.png')
print('Abbildungen erstellt:', len(created_figures))


In [ ]:
# Ergebnisse speichern
decision_rows = []
for _, row in primary_test_fixed_rank.head(30).iterrows():
    decision_rows.append({'target': row['target'], 'prefix_id': row['prefix_id'], 'scenario': row['scenario'], 'model': row['model'], 'test_pr_auc': row['pr_auc_average_precision'], 'test_roc_auc': row['roc_auc'], 'test_f1_fixed_05': row['f1'], 'precision_fixed_05': row['precision'], 'recall_fixed_05': row['recall'], 'brier_score': row['brier_score']})
model_decision_matrix = pd.DataFrame(decision_rows)
save_csv(model_decision_matrix, '21_model_decision_matrix_refinement.csv', index=False)
ablation_summary = ablation_impact.groupby(['scenario', 'model'], as_index=False).agg(mean_delta_pr_auc_vs_full=('delta_pr_auc_vs_full', 'mean'), min_delta_pr_auc_vs_full=('delta_pr_auc_vs_full', 'min'), max_delta_pr_auc_vs_full=('delta_pr_auc_vs_full', 'max'), mean_pr_auc=('pr_auc_average_precision', 'mean'), n=('pr_auc_average_precision', 'count')).sort_values(['model', 'mean_delta_pr_auc_vs_full'])
save_csv(ablation_summary, '22_ablation_summary_refinement.csv', index=False)
model_decision_matrix.head(10)
